# Aileron Design — As-Selected Re-Solve (COTS Servo)
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

NB4 (`aileron_design`) sized the ailerons against a *configured estimate*
of the servo (`config/aileron.yaml`: 12 g, 9g-class). NB11
(`cots_selection`) has since frozen a real part. This notebook **re-solves
the aileron design with the frozen servo** — as a *new, downstream*
notebook, so the design pipeline stays one-way (ADR-0001/ADR-0012): the
NB4 solution remains the record of what the estimate gave, this one is the
as-selected solution, and the delta between them is a finding against the
estimate.

What actually changes: the aileron *surface* sizing (geometry, flap
effectiveness, roll authority) does not depend on the servo at all, so it
is re-derived identically; the **hardware mass** (carved from the mass
fractions by the fuselage re-solve, NB14) and the **torque check** — now
against a real stall-torque rating instead of a class assumption — do.

## Inputs

- `out/components.yaml` *(NB11 — the frozen servo)*
- `out/airfoil.yaml`, `out/control_vanes.yaml` *(NB2/NB3, unchanged)*
- `out/aileron.yaml` *(NB4 — the conceptual solution, for the delta report)*
- `config/aileron.yaml`, `config/aerodynamics.yaml`

## Outputs

- `out/aileron_cots.yaml` *(same schema as `out/aileron.yaml` + frozen-servo
  block; consumed by NB14)*

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from dataclasses import replace

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design import FrozenComponents, reports
from conceptual_design.aileron_design import (
    AileronParams, size_aileron, vane_cruise_authority, write_aileron_yaml,
)

nb = nb_setup(style=False)   # no figures in this notebook
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` — same pattern as NB2–NB11, so this
notebook stays consistent with the upstream design state — and load the
wing/vane handoffs plus the frozen COTS hardware list.

---

In [ ]:
# -- Re-run the sizing loop + read the freeze and the NB4 estimate -------------
inp, result = solve_design_point(CONFIG_PATH)
ail_p = AileronParams.from_yaml(CONFIG_PATH / "aileron.yaml")

af      = load_handoff(OUT_PATH, "airfoil")
vanes   = load_handoff(OUT_PATH, "control_vanes")
ail_est = load_handoff(OUT_PATH, "aileron")
comps   = FrozenComponents.from_yaml(OUT_PATH / "components.yaml")
servo   = comps["servo"]

MTOW_kg  = result.m_total_kg
W_N      = MTOW_kg * inp.env.g
ddot_min = inp.aero.ddot_min_deg_s2

print(f"Converged MTOW : {MTOW_kg:.3f} kg  ({W_N:.2f} N)")
print(f"Wing            : b = {result.wing.b_wing*1e3:.0f} mm, "
      f"MAC = {result.wing.chord_mean*1e3:.1f} mm")
print(f"Frozen servo    : {servo.name}  ({servo.id})")
print(f"  mass           : {servo.mass_kg*1e3:.1f} g   (NB4 estimate: {ail_p.servo_mass_kg*1e3:.1f} g)")
print(f"  stall torque   : {servo.ratings['stall_torque_gcm']:.0f} g cm")
print(f"ddot_min        : {ddot_min:.0f} deg/s^2  (config/aerodynamics.yaml, shared with NB3/NB4)")

# Section 2 — Re-Solve with the Frozen Servo

Same physics as NB4 (`size_aileron`, one call), with the configured servo
mass estimate replaced by the frozen part's catalog mass. The residual
jet-vane roll authority in cruise is re-derived with the same
thrust-scaling law as NB4. The torque check is now a **real margin**:
catalog stall torque against the hinge-moment requirements of *both*
surfaces the single servo type drives (jet vanes, NB3; ailerons, this
notebook).

---

In [ ]:
# -- Re-solve NB4 with the frozen servo mass -----------------------------------
ail_p_cots = replace(ail_p, servo_mass_kg=servo.mass_kg)

ail = size_aileron(
    b_wing_m=result.wing.b_wing, chord_mean_m=result.wing.chord_mean,
    tc_ratio=af["tc_ratio"], AR=inp.aero.AR, e_oswald=af["e_oswald"],
    V_cruise=inp.mission.V_cruise, rho=inp.env.rho,
    I_roll=vanes["I_roll_kgm2"], p=ail_p_cots,
)

# residual jet-vane authority in cruise (same law as NB4)
auth = vane_cruise_authority(W_N, inp.prop.s_ratio, af["LD_cruise"], vanes)
ddot_roll_total = ail.ddot_roll_deg_s2 + auth.ddot_roll_cruise
cruise_ok = ddot_roll_total >= ddot_min

stall_gcm   = float(servo.ratings["stall_torque_gcm"])
margin_ail  = stall_gcm / ail.servo_torque_req_gcm
margin_vane = stall_gcm / vanes["servo_torque_req_gcm"]

print(f"Aileron chord / span  : {ail.c_aileron_m*1e3:.1f} / {ail.b_aileron_m*1e3:.1f} mm  "
      f"(geometry unchanged from NB4)")
print(f"Combined ddot_roll    : {ddot_roll_total:.1f} deg/s^2  (req {ddot_min:.0f})  "
      f"-> {'OK' if cruise_ok else 'FAIL'}")
print()
print(f"{'surface':<12}{'torque req [g cm]':>19}{'stall [g cm]':>14}{'margin':>9}")
print("-" * 54)
print(f"{'aileron':<12}{ail.servo_torque_req_gcm:>19.1f}{stall_gcm:>14.0f}{margin_ail:>8.1f}x")
print(f"{'jet vane':<12}{vanes['servo_torque_req_gcm']:>19.1f}{stall_gcm:>14.0f}{margin_vane:>8.1f}x")

# the frozen servo must actually drive both surfaces it was selected for
assert cruise_ok, "cruise roll authority lost in the re-solve -- upstream drift"
assert margin_ail >= 1.0 and margin_vane >= 1.0, (
    "frozen servo stall torque below a hinge-moment requirement -- "
    "the design has outgrown the frozen hardware")

# Section 3 — Output Export

`out/aileron_cots.yaml` — same schema as `out/aileron.yaml` (so the
fuselage re-solve consumes it identically) plus a `cots_servo` provenance
block.

---

In [ ]:
write_aileron_yaml(
    ail, ail_p_cots, OUT_PATH / "aileron_cots.yaml",
    ddot_roll_vane_cruise_deg_s2=auth.ddot_roll_cruise,
    ddot_min_deg_s2=ddot_min,
    regen_notebook="notebooks/aileron_design_cots.ipynb",
    extra={"cots_servo": {
        "id":             servo.id,
        "name":           servo.name,
        "mass_g":         round(servo.mass_kg * 1e3, 2),
        "stall_torque_gcm": stall_gcm,
        "margin_vs_aileron_req": round(margin_ail, 2),
        "margin_vs_vane_req":    round(margin_vane, 2),
    }},
)
print(f"As-selected aileron design written -> {OUT_PATH / 'aileron_cots.yaml'}")

# Section 4 — Delta vs. the Conceptual Solution (NB4) and Summary

The delta table is the point of the re-solve: what did freezing real
hardware change against the estimate the conceptual design used?

---

In [ ]:
hw_est  = 2 * (ail_est["servo_mass_kg_each"] + ail_est["linkage_mass_kg_each"]) * 1e3
hw_cots = 2 * (servo.mass_kg + ail_p.linkage_mass_kg) * 1e3

reports.print_aileron_cots_summary(ail, ail_est, servo, stall_gcm, hw_est,
                                   hw_cots, ddot_roll_total, margin_vane,
                                   margin_ail)